In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.metrics import (
    classification_report, roc_auc_score, balanced_accuracy_score,
    confusion_matrix, RocCurveDisplay
)
from xgboost import XGBClassifier

# ── Configuration ───────────────────────────────────────────────────
RESULTS_DIR = "results"
LIFESPAN_THRESHOLD = 10   # percent; compounds above this are "extenders"
MIN_DRUGS = 10            # GO terms must appear in at least this many drugs
N_FEATURES = 300          # top features to keep after SelectKBest
GO_CATEGORIES = ['biological_process', 'molecular_function']

os.makedirs(RESULTS_DIR, exist_ok=True)

# --------------------------------------------------
# STEP 1: LOAD DRUGAGE DATA (LABELS)
# --------------------------------------------------

drugage = pd.read_csv("C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\DrugAgeDataset\\drugage.csv")

drugage['compound_name'] = drugage['compound_name'].str.lower().str.strip()

drugage['avg_lifespan_change_percent'] = pd.to_numeric(
    drugage['avg_lifespan_change_percent'],
    errors='coerce'
)

compound_labels = (
    drugage
    .groupby('compound_name')['avg_lifespan_change_percent']
    .max()
    .reset_index()
)

compound_labels['label'] = compound_labels['avg_lifespan_change_percent'].apply(
    lambda x: 1 if x is not None and x > LIFESPAN_THRESHOLD else 0
)

print(f"DrugAge loaded: {drugage.shape}")
print(f"Label threshold: >{LIFESPAN_THRESHOLD}%")
print(f"Class distribution (all DrugAge): "
      f"{(compound_labels['label'] == 1).sum()} pos / "
      f"{(compound_labels['label'] == 0).sum()} neg")

# --------------------------------------------------
# STEP 2: LOAD DGIDB INTERACTIONS
# --------------------------------------------------

interactions = pd.read_csv("C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\DGIdb\\interactions.tsv", sep="\t")

interactions['drug_name'] = interactions['drug_name'].str.lower().str.strip()
interactions['gene_name'] = interactions['gene_name'].str.upper().str.strip()

drug_gene = interactions[['drug_name', 'gene_name']].drop_duplicates()

# Keep only drugs present in DrugAge
drug_gene = drug_gene[drug_gene['drug_name'].isin(drugage['compound_name'])]

print(f"DGIdb interactions (after DrugAge filter): {drug_gene.shape}")

# --------------------------------------------------
# STEP 3: LOAD GENE → GO ANNOTATIONS
# --------------------------------------------------

go = pd.read_csv("C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\GOterms_cElegans\\gene_go_annotations.tsv", sep="\t")

go['gene_name'] = go['gene_name'].str.upper().str.strip()
go['go_term'] = go['go_term'].str.strip()

# Filter to biologically informative GO categories
go = go[go['go_category'].isin(GO_CATEGORIES)]

print(f"GO annotations (after category filter): {go.shape}")

# --------------------------------------------------
# STEP 4: DRUG → GENE → GO MERGE
# --------------------------------------------------

drug_gene_go = drug_gene.merge(go, on="gene_name", how="inner")

print(f"Drug-Gene-GO table: {drug_gene_go.shape}")

# --------------------------------------------------
# STEP 5: BUILD GO FEATURE MATRIX WITH DIMENSIONALITY REDUCTION
# --------------------------------------------------

# 5a: Binary crosstab (presence/absence, not raw counts)
X = pd.crosstab(
    drug_gene_go['drug_name'],
    drug_gene_go['go_term']
)
X = (X > 0).astype(int)

print(f"Raw feature matrix: {X.shape}")

# 5b: Align with labels
X = X.loc[X.index.isin(compound_labels['compound_name'])]
y = compound_labels.set_index('compound_name').loc[X.index]['label']

print(f"After label alignment: X={X.shape}, y={y.shape}")
print(f"Class distribution (pipeline): "
      f"{(y == 1).sum()} pos / {(y == 0).sum()} neg "
      f"(ratio {(y == 1).sum() / max((y == 0).sum(), 1):.2f}:1)")

# 5c: Remove rare GO terms
col_sums = X.sum(axis=0)
X = X.loc[:, col_sums >= MIN_DRUGS]
print(f"After min_drugs={MIN_DRUGS} filter: {X.shape}")

# 5d: Remove near-zero-variance features
vt = VarianceThreshold(threshold=0.01)
X_filtered = vt.fit_transform(X)
kept_columns = X.columns[vt.get_support()]
X = pd.DataFrame(X_filtered, index=X.index, columns=kept_columns)
print(f"After variance filter: {X.shape}")

# 5e: SelectKBest using mutual information
if X.shape[1] > N_FEATURES:
    skb = SelectKBest(mutual_info_classif, k=N_FEATURES)
    X_selected = skb.fit_transform(X, y)
    kept_columns = X.columns[skb.get_support()]
    X = pd.DataFrame(X_selected, index=X.index, columns=kept_columns)
    print(f"After SelectKBest (k={N_FEATURES}): {X.shape}")

# 5f: Sanitize column names for XGBoost compatibility
X.columns = X.columns.str.replace(r'[\[\]<>]', '', regex=True)

print(f"\nFinal dataset: X={X.shape}, y={y.shape}")
print(f"Sparsity: {(X == 0).sum().sum() / (X.shape[0] * X.shape[1]) * 100:.1f}%")

print("\nSample rows (X):")
print(X.head(10))
print("\nCorresponding labels (y):")
print(y.head(10))

# --------------------------------------------------
# SAVE DATASET TO CSV
# --------------------------------------------------

dataset = X.copy()
dataset['label'] = y

output_path = "C:\\Users\\flori\\Documents\\GitHub\\Ora-Biomedical-Student-Machine-Learning-Project\\data\\ndataset.csv"
dataset.to_csv(output_path)
print(f"\nDataset saved to {output_path} ({dataset.shape[0]} rows, {dataset.shape[1]} columns)")


DrugAge loaded: (3423, 15)
Label threshold: >10%
Class distribution (all DrugAge): 582 pos / 464 neg
DGIdb interactions (after DrugAge filter): (4030, 2)
GO annotations (after category filter): (141451, 5)
Drug-Gene-GO table: (195905, 6)
Raw feature matrix: (343, 8423)
After label alignment: X=(343, 8423), y=(343,)
Class distribution (pipeline): 209 pos / 134 neg (ratio 1.56:1)
After min_drugs=10 filter: (343, 2577)
After variance filter: (343, 2577)
After SelectKBest (k=300): (343, 300)

Final dataset: X=(343, 300), y=(343,)
Sparsity: 90.8%

Sample rows (X):
go_term                  (S)-limonene 6-monooxygenase activity  \
drug_name                                                        
2,4-dinitrophenol                                            0   
3,4-dichloroisocoumarin                                      1   
3-methoxycatechol                                            0   
6-hydroxyflavone                                             0   
7,8-dihydroxyflavone                  

In [6]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    balanced_accuracy_score, f1_score
)
import numpy as np

# --------------------------------------------------
# TRAIN / TEST SPLIT (Held-out test set)
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,          # 20% held-out test set
    stratify=y,
    random_state=42
)

print("\nTrain/Test Split:")
print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")
print(f"Train class distribution: {(y_train == 1).sum()} pos / {(y_train == 0).sum()} neg")
print(f"Test class distribution:  {(y_test == 1).sum()} pos / {(y_test == 0).sum()} neg")


Train/Test Split:
Train size: 274
Test size:  69
Train class distribution: 167 pos / 107 neg
Test class distribution:  42 pos / 27 neg


In [7]:
# --------------------------------------------------
# RANDOM FOREST MODEL
# --------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# --------------------------------------------------
# STRATIFIED 5-FOLD CROSS VALIDATION
# --------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'roc_auc': 'roc_auc',
    'balanced_accuracy': 'balanced_accuracy',
    'f1': 'f1'
}

cv_results = cross_validate(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("\nCross-Validation Results (Training Set Only):")
for metric in scoring.keys():
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.3f} ± {scores.std():.3f}")


Cross-Validation Results (Training Set Only):
roc_auc: 0.549 ± 0.058
balanced_accuracy: 0.513 ± 0.036
f1: 0.656 ± 0.043


In [8]:
# --------------------------------------------------
# FINAL MODEL TRAINING
# --------------------------------------------------

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=500, n_jobs=-1,
                       random_state=42)

In [9]:
# --------------------------------------------------
# TEST SET EVALUATION
# --------------------------------------------------

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

test_roc = roc_auc_score(y_test, y_proba)
test_bal_acc = balanced_accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

print("\nHeld-Out Test Set Performance:")
print(f"ROC-AUC:           {test_roc:.3f}")
print(f"Balanced Accuracy: {test_bal_acc:.3f}")
print(f"F1 Score:          {test_f1:.3f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred))


Held-Out Test Set Performance:
ROC-AUC:           0.664
Balanced Accuracy: 0.579
F1 Score:          0.690

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.50      0.44      0.47        27
           1       0.67      0.71      0.69        42

    accuracy                           0.61        69
   macro avg       0.58      0.58      0.58        69
weighted avg       0.60      0.61      0.60        69



In [10]:
from sklearn.model_selection import cross_val_score

rf_full = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

scores = cross_val_score(
    rf_full,
    X,   # full dataset
    y,
    cv=5,
    scoring="roc_auc"
)

print("Full 5-fold CV ROC-AUC:", scores.mean(), "±", scores.std())

Full 5-fold CV ROC-AUC: 0.6224201799405051 ± 0.047093458037277214


In [11]:
import pandas as pd

rf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
top_20 = importances.sort_values(ascending=False).head(20)

print(top_20)

go_term
GTP binding                                           0.015293
monoatomic ion transmembrane transport                0.014957
negative regulation of DNA-templated transcription    0.013745
defense response to bacterium                         0.013511
icosanoid biosynthetic process                        0.012553
protein kinase activity                               0.010992
protein kinase binding                                0.010709
cellular response to mechanical stimulus              0.010299
response to oxidative stress                          0.010177
transcription cis-regulatory region binding           0.009891
protein-containing complex binding                    0.009529
nervous system development                            0.009463
angiogenesis                                          0.009191
response to cold                                      0.009059
transmembrane transporter activity                    0.008937
adaptive immune response                       

In [12]:
top_features = top_20.index

for term in top_features[:10]:
    mean_pos = X.loc[y==1, term].mean()
    mean_neg = X.loc[y==0, term].mean()
    print(term)
    print("  pos freq:", round(mean_pos,3))
    print("  neg freq:", round(mean_neg,3))
    

GTP binding
  pos freq: 0.1
  neg freq: 0.179
monoatomic ion transmembrane transport
  pos freq: 0.254
  neg freq: 0.246
negative regulation of DNA-templated transcription
  pos freq: 0.469
  neg freq: 0.373
defense response to bacterium
  pos freq: 0.134
  neg freq: 0.015
icosanoid biosynthetic process
  pos freq: 0.177
  neg freq: 0.06
protein kinase activity
  pos freq: 0.311
  neg freq: 0.172
protein kinase binding
  pos freq: 0.426
  neg freq: 0.336
cellular response to mechanical stimulus
  pos freq: 0.234
  neg freq: 0.067
response to oxidative stress
  pos freq: 0.354
  neg freq: 0.328
transcription cis-regulatory region binding
  pos freq: 0.483
  neg freq: 0.418


In [13]:
# Compute positive/negative frequencies for top features
top_features = top_20.index

enrichment_data = []
for term in top_features:
    pos_freq = X.loc[y==1, term].mean()
    neg_freq = X.loc[y==0, term].mean()
    enrichment_data.append({
        'GO Term': term,
        'Feature Importance': importances[term],
        'Positive Freq': pos_freq,
        'Negative Freq': neg_freq,
        'Enrichment (Pos - Neg)': pos_freq - neg_freq
    })

enrichment_df = pd.DataFrame(enrichment_data)
enrichment_df = enrichment_df.sort_values(by='Feature Importance', ascending=False)

# Display table
enrichment_df.reset_index(drop=True, inplace=True)
print(enrichment_df)

                                              GO Term  Feature Importance  \
0                                         GTP binding            0.015293   
1              monoatomic ion transmembrane transport            0.014957   
2   negative regulation of DNA-templated transcrip...            0.013745   
3                       defense response to bacterium            0.013511   
4                      icosanoid biosynthetic process            0.012553   
5                             protein kinase activity            0.010992   
6                              protein kinase binding            0.010709   
7            cellular response to mechanical stimulus            0.010299   
8                        response to oxidative stress            0.010177   
9         transcription cis-regulatory region binding            0.009891   
10                 protein-containing complex binding            0.009529   
11                         nervous system development            0.009463   

# Strongly Enriched in Lifespan-Extending Compounds

| GO Term                                         | Enrichment (Pos-Neg) | Feature Importance |
| ----------------------------------------------- | -------------------- | ------------------ |
| Cellular response to mechanical stimulus        | 0.167                | 0.0103             |
| Positive regulation of interleukin-6 production | 0.154                | 0.0086             |
| Protein kinase activity                         | 0.139                | 0.0110             |
| Defense response to bacterium                   | 0.119                | 0.0135             |
| Icosanoid biosynthetic process                  | 0.117                | 0.0126             |
| Transmembrane transporter activity              | 0.108                | 0.0089             |


# More Frequent in Non-Entending Compounds

| GO Term                           | Enrichment | Feature Importance |
| --------------------------------- | ---------- | ------------------ |
| GTP binding                       | -0.079     | 0.0153             |
| Response to cold                  | -0.086     | 0.0091             |
| Nuclear steroid receptor activity | -0.047     | 0.0082             |
